In [ ]:
import os
import pandas as pd
import torch
import numpy as np

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

def load_dataset(path):
    data = pd.read_csv(path)
    # Sensitive attribute (keep only is_male)
    S = data["is_male"].values.astype(int)

    # Target
    y = data["hospital_expire_flag"].values.astype(int)

    # Drop unused target, treatment, and other sensitive attributes
    drop_columns = [
        "hospital_expire_flag", "thirtyday_expire_flag", "icu_los", "hosp_los",  # targets
        "vent", "rrt",  # treatments
        "is_male", "age",  # sensitive attributes
        "race_white", "race_black", "race_hispanic", "race_other"  # other sensitive attrs
    ]

    # Drop with safeguard for missing columns
    X_raw = data.drop(columns=[col for col in drop_columns if col in data.columns])

    # Identify numeric and categorical features
    numerical_features = X_raw.select_dtypes(include=["float64", "int64"]).columns.tolist()
    categorical_features = X_raw.select_dtypes(include=["object", "bool", "category"]).columns.tolist()


    # Transformers
    num_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", MinMaxScaler())
    ])

    cat_transformer = Pipeline([
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_transformer, numerical_features),
        ("cat", cat_transformer, categorical_features)
    ])

    # Apply transformation
    X_processed = preprocessor.fit_transform(X_raw)

    # Feature names
    num_names = numerical_features
    if categorical_features:
        cat_names = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)
        feature_names = list(num_names) + list(cat_names)
    else:
        feature_names = list(num_names)

    # Convert to DataFrame
    X_df = pd.DataFrame(X_processed, columns=feature_names)
    y = 1-y

    return X_df, y, S


In [ ]:
from sklearn.model_selection import train_test_split
# Step 1: Load data
X_df, y, S = load_dataset()
print(sum(y==0))
print(sum(S==1))

print(f"Loaded {X_df.shape[0]} samples with {X_df.shape[1]} features.")
print(f"Target positive rate: {y.mean():.2%}, Sensitive attribute male: {S.mean():.2%}")

X_aug = X_df.copy()
X_aug["S"] = S

# # Step 2: Split
X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
    X_aug, y, S, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Define your model
class SmallNet(nn.Module):
    def __init__(self, input_dim):
        super(SmallNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 124),
            nn.ReLU(),
            nn.Linear(124,64),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(64, 1),  # outputs logits
        )
        
    def forward(self, x):
        return self.model(x)

In [ ]:
x_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # make it (N, 1)
s_train_tensor = torch.tensor(S_train, dtype=torch.long)

x_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
s_test_tensor = torch.tensor(S_test, dtype=torch.long)

In [ ]:
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
input_dim = X_train.shape[1]
model = SmallNet(input_dim)

criterion = nn.BCEWithLogitsLoss()  # since output is logits
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=0.001)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)
    avg_loss = epoch_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(x_test_tensor)
    probs_test = torch.sigmoid(logits)
    preds = (probs_test >= 0.5).float()
    acc = (preds.eq(y_test_tensor)).sum().item() / y_test_tensor.size(0)
    print("Test accuracy:", acc)

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(x_train_tensor)
    probs_train = torch.sigmoid(logits)
    preds = (probs_train >= 0.5).float()
    acc = (preds.eq(y_train_tensor)).sum().item() / y_train_tensor.size(0)
    print("Train accuracy:", acc)

In [ ]:
# %%
# Save train/test datasets with predicted probabilities

# Put model in eval mode
model.eval()
with torch.no_grad():
    # Train probabilities
    train_logits = model(x_train_tensor)
    train_probs = torch.sigmoid(train_logits).cpu().numpy().flatten()

    # Test probabilities
    test_logits = model(x_test_tensor)
    test_probs = torch.sigmoid(test_logits).cpu().numpy().flatten()

# Recreate DataFrames
train_df = X_train.copy()
train_df=train_df.add_prefix("preprocessed_")
train_df=train_df.rename(columns={"preprocessed_S":"G"})
train_df["y"] = y_train
train_df["prob_Y"] = train_probs


test_df = X_test.copy()
test_df=test_df.add_prefix("preprocessed_")
test_df=test_df.rename(columns={"preprocessed_S":"G"})
test_df["y"] = y_test
test_df["prob_Y"] = test_probs


# Save to CSV
train_path = "data/mimic/mimic_train_with_prob.csv"
test_path = "data/mimic/mimic_test_with_prob.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved train set to {train_path}")
print(f"Saved test set to {test_path}")
